In [5]:
from datasets import load_dataset
import random
from collections import defaultdict
import regex as re
from __future__ import annotations

ds = load_dataset("roneneldan/TinyStories")['train']
text = "<|endoftext|>".join(ds[0:10000]['text'])
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
special_tokens = ["<|endoftext|>"]
vocab_limit = 1000



In [6]:
# return counts of pre_tokens as a dictionary
def pre_tokenization(text: str, PAT: str, special_tokens: list[str]) -> dict[str]:
    i = 0
    pre_tokens_str = defaultdict(int)
    st_PAT = "|".join([re.escape(x) for x in special_tokens])
    for st_match in re.finditer(st_PAT, text):
        j = st_match.start()
        ss = text[i:j]
        for match in re.finditer(PAT, ss):
            word = match.group()
            pre_tokens_str[word] += 1
        i = st_match.end()
    ss = text[i:]
    for match in re.finditer(PAT, ss):
        word = match.group()
        pre_tokens_str[word] += 1
    return pre_tokens_str



pre_tokens = pre_tokenization(text, PAT, special_tokens)
print(len(pre_tokens))

12085


In [7]:
# update bp count and pre_tokens as a result of bp merge
def update_bp_count(bp, bp_count, pre_tokens):
    for ini_word in list(pre_tokens.keys()):
        i = 0
        count = pre_tokens[ini_word]
        word = ini_word
        # scan through the word to look for the bp. if found, update the word and bp_count influenced by the merge
        while i < len(word)-1:
            if word[i]==bp[0] and word[i+1]==bp[1]:
                if i > 0:
                    bp_count[(word[i-1], bp[0]+bp[1])] += count
                    bp_count[(word[i-1], bp[0])] -= count
                if i+2 < len(word):
                    bp_count[(bp[0]+bp[1], word[i+2])] += count
                    bp_count[(bp[1], word[i+2])] -= count
                bp_count[(bp[0], bp[1])] -= count
                word = word[:i] + (bp[0]+bp[1],) + word[i+2:]
            i += 1

        # update tokenization of the pretoken / word
        if word != ini_word:
            pre_tokens.pop(ini_word)
            pre_tokens[word] = count
            


def construct_bpe(pre_tokens_str: dict[str],  vocab_size: int, special_tokens: list[str]):
    pre_tokens = dict()
    for word, count in pre_tokens_str.items():
        word_tuple_of_bytes = tuple(bytes([a]) for a in word.encode("utf-8"))
        pre_tokens[word_tuple_of_bytes] = count
    
    # construct initial vocab
    token_id = 0
    vocab = dict()
    for i in range(256):
        vocab[token_id] = bytes([i])
        token_id += 1
    for t in special_tokens:
        vocab[token_id] = t.encode("utf-8")
        token_id += 1
    

    # construct initial bytes-pair count
    bp_count = defaultdict(int)
    for token, count in pre_tokens.items():
        for i in range(len(token)-1):
            bp = (token[i], token[i+1])
            bp_count[bp] += count

    # merge
    merges = []
    while len(vocab) < vocab_size:
        # find the most frequent bp, if there is a tie then pick the first based on lexi order
        max_count = max(bp_count.values())
        tmp = [bp for bp, count in bp_count.items() if count==max_count]
        bp = max(tmp)
        merges.append(bp)

        # add it to vocab
        vocab[token_id] = bp[0] + bp[1]
        token_id += 1

        # update bp count and tokenization of pre-tokens
        update_bp_count(bp, bp_count, pre_tokens)
        assert(bp_count[bp]==0)
        
    return vocab, merges


vocab, merges = construct_bpe(pre_tokens, vocab_limit, special_tokens)